# Firasa, Dashboard Export

This notebook prepares the one file the HTML dashboard actually reads: a single table with every test customer's fused risk score, risk level, recommended action, a stable display name, and the top reasons behind their score from both branches.

Nothing here is retrained. Every model is loaded from the artifacts `Firasa_03_Branch1_XGBoost_LightGBM.ipynb` and `Firasa_04_Branch2_GRU_Fusion.ipynb` already saved, XGBoost, the GRU, its scaler and encoder, and the fusion meta-model. This notebook's only job is to run those saved models once over the test set and export the result.

**Output:** `customer_scores_with_shap.csv`, one row per test customer (about 8,297 rows), ready to drop into the dashboard's `data/` folder.

## Step 1: Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json, hashlib
import numpy as np
import pandas as pd
import joblib

import tensorflow as tf
from tensorflow import keras
import xgboost as xgb

pd.set_option('display.max_columns', 50)

DATA_DIR = '/content/drive/MyDrive/Firasa Project'
TEST_FILE = f'{DATA_DIR}/amex_test.parquet'
AGG_TEST_FILE = f'{DATA_DIR}/amex_agg_test.parquet'

ARTIFACT_DIR = '/content/drive/MyDrive/firasa_artifacts'
BRANCH1_FILE = f'{ARTIFACT_DIR}/branch1_scores.csv'
BRANCH1_MODEL_PATH = f'{ARTIFACT_DIR}/branch1_xgboost.json'
BRANCH1_CONFIG_PATH = f'{ARTIFACT_DIR}/branch1_config.json'

GRU_MODEL_PATH = f'{ARTIFACT_DIR}/branch2_gru.keras'
SCALER_PATH = f'{ARTIFACT_DIR}/branch2_scaler.joblib'
ENCODER_PATH = f'{ARTIFACT_DIR}/branch2_ordinal_encoder.joblib'
META_PATH = f'{ARTIFACT_DIR}/fusion_meta_model.joblib'
FUSION_CONFIG_PATH = f'{ARTIFACT_DIR}/branch2_fusion_config.json'

EXPORT_PATH = f'{ARTIFACT_DIR}/customer_scores_with_shap.csv'

SEED = 42
np.random.seed(SEED)

Every saved file is loaded here, once. Branch 1 (XGBoost) and Branch 2 (GRU) load their trained weights directly; the scaler, encoder, and fusion meta-model load the exact objects fitted on train in Firasa_04, so test customers get transformed the same way they were during evaluation, not refit.

In [3]:
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(BRANCH1_MODEL_PATH)
with open(BRANCH1_CONFIG_PATH) as f:
    b1_config = json.load(f)
B1_FEATURES = b1_config['features']

gru_model = keras.models.load_model(GRU_MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
ord_enc = joblib.load(ENCODER_PATH)
meta = joblib.load(META_PATH)
with open(FUSION_CONFIG_PATH) as f:
    fusion_config = json.load(f)

SEQ_COLS = fusion_config['seq_cols']
CAT_COLS = fusion_config['cat_cols']
MAX_LEN = fusion_config['max_len']
RISK_BINS = fusion_config['risk_bins']
RISK_LEVELS = fusion_config['risk_levels']
# Firasa_04's RISK_ACTIONS was written for a technical/case-manager audience.
# The dashboard is meant for a general, non-technical viewer, so the wording
# is overridden here, in plain, easy-to-read English, without needing to
# retrain or re-save anything upstream. Everything else from fusion_config
# (bins, colors, thresholds) stays exactly as Firasa_04 froze it.
RISK_ACTIONS = {
    'Low':      'No action needed. Keep monitoring as usual.',
    'Moderate': 'Add to the watchlist and send a friendly payment reminder.',
    'High':     'Reach out proactively and offer a flexible payment plan.',
    'Critical': 'Urgent: contact the customer right away and review their account together.',
}
LEVEL_COLORS = fusion_config['level_colors']
THRESHOLDS = fusion_config['thresholds']

print(f"Loaded XGBoost ({len(B1_FEATURES)} features), GRU (max_len={MAX_LEN}), "
      f"scaler, encoder, and the fusion meta-model.")

Loaded XGBoost (1802 features), GRU (max_len=13), scaler, encoder, and the fusion meta-model.


## Step 2: Load Test Customers and Branch 1's Scores

Branch 1's predictions were already computed and saved by `Firasa_03`, so this reads them straight from `branch1_scores.csv` rather than rescoring XGBoost. Only the last-statement checkpoint is kept here, since the dashboard shows each customer's current risk, not the 3 and 6-month early-warning view the modelling notebooks focus on.

In [4]:
test_df = pd.read_parquet(TEST_FILE)
test_df = test_df.sort_values(['customer_ID', 'S_2']).reset_index(drop=True)
test_df[CAT_COLS] = ord_enc.transform(test_df[CAT_COLS].astype(str))
test_df[SEQ_COLS] = test_df[SEQ_COLS].astype('float32')

branch1 = pd.read_csv(BRANCH1_FILE)
b1_test = branch1[(branch1['split'] == 'test') & (branch1['months_before_last'] == 0)]
b1_test = b1_test[['customer_ID', 'target', 'p_xgb']].reset_index(drop=True)

print(f"Test customers: {test_df['customer_ID'].nunique()}")
print(f"Branch 1 last-statement scores: {len(b1_test)} rows")

Test customers: 8297
Branch 1 last-statement scores: 8297 rows


## Step 3: Rebuild Test Sequences and Score Branch 2

Same padding logic as Firasa_04's Step 5: sequences are padded at the front, so the last real statement always lands in the same array position, and the GRU's last-statement prediction is read from that fixed position. The scaler here only transforms, it was already fit on train customers in Firasa_04.

In [5]:
test_df[SEQ_COLS] = scaler.transform(test_df[SEQ_COLS]).astype('float32')

def build_sequences(df):
    seqs, ids = [], []
    for cid, g in df.groupby('customer_ID', sort=True):
        seqs.append(g[SEQ_COLS].to_numpy(dtype='float32'))
        ids.append(cid)
    X = keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAX_LEN, dtype='float32', padding='pre', truncating='pre', value=0.0)
    return X, np.array(ids, dtype=object)

X_test_seq, ids_test = build_sequences(test_df)

p_branch2 = gru_model.predict(X_test_seq, batch_size=1024, verbose=0)[:, MAX_LEN - 1, 0]
b2_test = pd.DataFrame({'customer_ID': ids_test, 'p_branch2': p_branch2})

print(f"Branch 2 scored {len(b2_test)} test customers at the last statement.")

Branch 2 scored 8297 test customers at the last statement.


## Step 4: Fuse the Two Branches and Assign Risk Levels

The loaded meta-model combines both branches exactly as it did during evaluation, and the risk bins are the same ones frozen on validation in Firasa_04, not re-tuned here.

In [6]:
scores = b1_test.merge(b2_test, on='customer_ID', how='inner')
assert len(scores) == len(b1_test) == len(b2_test), "Branch 1 and Branch 2 scored different customers."

eps = 1e-6
p = lambda col: np.clip(scores[col].values, eps, 1 - eps)
logit = lambda x: np.log(x / (1 - x))
Z = np.column_stack([logit(p('p_xgb')), logit(p('p_branch2'))])
scores['p_fused'] = meta.predict_proba(Z)[:, 1]

scores['risk_level'] = pd.cut(scores['p_fused'], bins=[-np.inf] + RISK_BINS + [np.inf],
                              labels=RISK_LEVELS, right=False)
scores['action'] = scores['risk_level'].map(RISK_ACTIONS)

print(scores['risk_level'].value_counts().reindex(RISK_LEVELS))

risk_level
Low         4696
Moderate     705
High         856
Critical    2040
Name: count, dtype: int64


## Step 5: Display Names

A deterministic pseudonym for each customer: the same `customer_ID` always produces the same name, since the name is derived from a hash of the ID rather than assigned randomly each run. Names are real Saudi/Gulf first and family names, written in English letters.

In [7]:
FIRST_NAMES = [
    # Female
    "Amal", "Sarah", "Asma", "Jameelah", "Shahad", "Safaa", "Aisha", "Kholoud",
    "Noor", "Noura", "Layan", "Abeer", "Bayan", "Duha", "Dalia", "Hessa",
    "Munira", "Maha", "Shorouq", "Dina", "Dima", "Nada", "Ruaa", "Nuhal",
    "Rose", "Maysaa", "Farah", "Haya", "Hadeel", "Lulwa", "Ghaidaa", "Safia",
    "Lina", "Noura", "Fatimah", "Reem", "Lama", "Hind", "Alia", "Leen",
    "Rana", "Jood", "Mona", "Marwa", "Hasnaa", "Dana",
    # Male
    "Abdulaziz", "Abdulrahman", "Khalid", "Hassan", "Emad", "Mohammed",
    "Muayad", "Ali", "Fahad", "Turki", "Saleh", "Abdullah", "Ahmed", "Fares",
    "Ziad", "Ghaith", "Laith", "Ammar", "Omar", "Talal", "Faisal", "Saif",
    "Sanad", "Nayef", "Nawaf", "Nasser", "Yasser", "Rashed", "Salman",
    "Sultan", "Ibrahim", "Saad", "Saeed", "Mishal", "Saud", "Bandar", "Badr",
    "Majed", "Malik", "Rakan", "Rayan", "Abdulmohsen", "Anas", "Eyad",
    "Fawaz", "Adel", "Tamim", "Zaid", "Mishari", "Hattan", "Yousef",
    "Sattam", "Marwan", "Abdulilah",
]
LAST_NAMES = [
    "Alzahrani", "Alghamdi", "Alqahtani", "Alotaibi", "Alharbi", "Alshehri",
    "Almalki", "Aldosari", "Alanazi", "Alsubaie", "Alsulami", "Alomari",
    "Alkhalidi", "Albalawi", "Alrasheed", "Almutairi", "Alhamdan",
    "Alshahrani", "Alahmari", "Alasmari", "Alnasser", "Alhumaidi", "Alkinani",
    "Albashiri", "Alshammari", "Alyami", "Aljuhani", "Alhazmi", "Althaqafi",
    "Aleisa", "Alfawzan", "Alzamel", "Alshamrani", "Almarri", "Aldakheel",
    "Alsudairi", "Alqusaibi", "Althunayan", "Almuhaidib", "Altamimi",
]

# Manual overrides checked before the hash. Useful for a demo: pick one real
# customer_ID from the test set and pin a specific name to it, e.g. your own,
# for a presentation, without touching the random assignment for anyone else.
NAME_OVERRIDES = {
     '0037e0080bb0973f46cee0aa17c50fdfca769470af6bfae29b7b53f6d3848acd': 'Noura Mesfer Alzahrani',
     '002d6c879b55da7dca6b7ae8bb5054a8ab9c5b24e0b15717f46c900f01977e59': 'Abeer Mohammed Alshahrani',
     '00130d73f6ebd4c0066dd5ec5c290378ff29d4fc62c17735f57f88cf3ddbb1e7': 'Layan Fahad Alhumaidi',

}

# Father names: reuses the 54 male first names above (a Saudi man's own first
# name commonly also serves as a father's name), plus names specific to this
# middle-name role. Combined with FIRST_NAMES x LAST_NAMES, a three-part name
# ("Noura Fahad Alzahrani") multiplies the combination count instead of adding
# to it, which is what actually brings collisions down, no visible number
# needed next to a repeated name.
FATHER_NAMES = [
    "Abdulaziz", "Abdulrahman", "Khalid", "Hassan", "Emad", "Mohammed",
    "Muayad", "Ali", "Fahad", "Turki", "Saleh", "Abdullah", "Ahmed", "Fares",
    "Ziad", "Ghaith", "Laith", "Ammar", "Omar", "Talal", "Faisal", "Saif",
    "Sanad", "Nayef", "Nawaf", "Nasser", "Yasser", "Rashed", "Salman",
    "Sultan", "Ibrahim", "Saad", "Saeed", "Mishal", "Saud", "Bandar", "Badr",
    "Majed", "Malik", "Rakan", "Rayan", "Abdulmohsen", "Anas", "Eyad",
    "Fawaz", "Adel", "Tamim", "Zaid", "Mishari", "Hattan", "Yousef",
    "Sattam", "Marwan", "Abdulilah",
    "Mesfer", "Ayedh", "Awad", "Masoud", "Abdullatif", "Hamad", "Hamdan",
    "Mansour", "Ghanem", "Obaid", "Jaber", "Munif", "Mubarak", "Wafi",
    "Salem", "Tariq", "Awwad", "Waleed", "Safar", "Hazza", "Ghazi",
    "Mutlaq", "Matar",
]

def customer_display_name(customer_id):
    if customer_id in NAME_OVERRIDES:
        return NAME_OVERRIDES[customer_id]
    h = hashlib.md5(str(customer_id).encode()).hexdigest()
    first_idx = int(h[:8], 16) % len(FIRST_NAMES)
    father_idx = int(h[8:16], 16) % len(FATHER_NAMES)
    last_idx = int(h[16:24], 16) % len(LAST_NAMES)
    return f"{FIRST_NAMES[first_idx]} {FATHER_NAMES[father_idx]} {LAST_NAMES[last_idx]}"

scores['display_name'] = scores['customer_ID'].apply(customer_display_name)

print(f"Unique names: {scores['display_name'].nunique()} across {scores['customer_ID'].nunique()} customers")
scores[['customer_ID', 'display_name', 'risk_level']].head(20)

Unique names: 8207 across 8297 customers


,customer_ID,display_name,risk_level
0,0005ef1480ffe4f0e5908eae07db8e05f2e705bca6a57d...,Asma Marwan Alshamrani,Critical
1,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,Duha Anas Althunayan,Critical
2,00130d73f6ebd4c0066dd5ec5c290378ff29d4fc62c177...,Layan Fahad Alhumaidi,Low
3,0017039b6f3eceb538de3d312f77028bf274cf61f992a3...,Sattam Ghazi Aleisa,Moderate
4,001ff5350ab524ad281b84d191c1d196bb575935ee0bb7...,Aisha Tamim Alkinani,Critical
5,002d6c879b55da7dca6b7ae8bb5054a8ab9c5b24e0b157...,Abeer Mohammed Alshahrani,Low
6,0037e0080bb0973f46cee0aa17c50fdfca769470af6bfa...,Noura Mesfer Alzahrani,Low
7,003f56dff9ddbe1ea99f99041e116f302b52abcfe4640f...,Mohammed Badr Albashiri,Critical
8,004af80f3576faf52144aa09d9f2fe30f8861345fc005b...,Safia Mesfer Alnasser,Moderate
9,004f0a4ff711d3f9992645cc7d25f93cb0517de7d73029...,Marwan Sattam Almuhaidib,Critical


**Insight:** the display name is now three parts, first name, father's name, family name, matching how Saudi names are naturally written, not a visible workaround. With 100 first names, 77 father names, and 40 family names, there are 308,000 combinations for roughly 8,297 customers. Using the birthday-paradox approximation (expected colliding pairs is roughly n(n-1) / 2N), that works out to about 112 colliding pairs, so around 223 customers (2.7%) are expected to share their full three-part name with exactly one other customer, and effectively none with more than one. No number ever needs to appear next to a name for this to hold.


## Step 6: SHAP Reasons for Every Customer (Branch 1)

Firasa_04's Step 15 computed SHAP for just 4 example customers, one per risk level, since that was enough for a design mockup. The dashboard needs this for every customer who might actually be opened, so this runs `TreeExplainer` once over the full test set. `TreeExplainer` is fast enough on XGBoost that this is a single vectorized call, not a loop.

In [8]:
agg_test = pd.read_parquet(AGG_TEST_FILE)
if 'customer_ID' in agg_test.columns:
    agg_test = agg_test.set_index('customer_ID')
X_b1_test = agg_test.loc[scores['customer_ID'], B1_FEATURES]

import shap
explainer_b1 = shap.TreeExplainer(xgb_model)
shap_values_b1 = explainer_b1.shap_values(X_b1_test)

print(f"SHAP computed for {shap_values_b1.shape[0]} customers x {shap_values_b1.shape[1]} features.")

SHAP computed for 8297 customers x 1802 features.


In [9]:
def top_reasons(shap_row, feature_names, n=3):
    top_idx = np.argsort(np.abs(shap_row))[::-1][:n]
    parts = []
    for i in top_idx:
        direction = 'higher risk' if shap_row[i] > 0 else 'lower risk'
        parts.append(f"{feature_names[i]} -> {direction}")
    return '; '.join(parts)

xgb_reasons = [top_reasons(shap_values_b1[i], B1_FEATURES) for i in range(len(X_b1_test))]
scores = scores.merge(
    pd.DataFrame({'customer_ID': X_b1_test.index, 'xgb_top_reasons': xgb_reasons}),
    on='customer_ID', how='left')

scores[['customer_ID', 'display_name', 'xgb_top_reasons']].head(3)

,customer_ID,display_name,xgb_top_reasons
0,0005ef1480ffe4f0e5908eae07db8e05f2e705bca6a57d...,Asma Marwan Alshamrani,P_2_last -> higher risk; D_39_last -> higher r...
1,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,Duha Anas Althunayan,P_2_last -> higher risk; D_39_last -> higher r...
2,00130d73f6ebd4c0066dd5ec5c290378ff29d4fc62c177...,Layan Fahad Alhumaidi,P_2_last -> lower risk; P_2_min -> lower risk;...


## Step 7: GRU Saliency Reasons for Every Customer (Branch 2)

Same Grad x Input method as Firasa_04's Step 12 and Step 15, run once over every test customer's sequence rather than 4 examples. This is one batched gradient computation, not a loop over customers.

In [10]:
x_all = tf.convert_to_tensor(X_test_seq, dtype=tf.float32)

with tf.GradientTape() as tape:
    tape.watch(x_all)
    preds = gru_model(x_all, training=False)[:, MAX_LEN - 1, 0]
grads = tape.gradient(preds, x_all).numpy()
grad_x_input = grads * X_test_seq   # (n_customers, MAX_LEN, n_features), last real month = MAX_LEN - 1

last_month = grad_x_input[:, MAX_LEN - 1, :]   # each customer's own most recent statement

def top_gru_reasons(row, feature_names, n=3):
    top_idx = np.argsort(np.abs(row))[::-1][:n]
    parts = []
    for i in top_idx:
        direction = 'higher risk' if row[i] > 0 else 'lower risk'
        parts.append(f"{feature_names[i]} -> {direction}")
    return '; '.join(parts)

gru_reasons = [top_gru_reasons(last_month[i], SEQ_COLS) for i in range(len(ids_test))]
gru_reasons_df = pd.DataFrame({'customer_ID': ids_test, 'gru_top_reasons': gru_reasons})
scores = scores.merge(gru_reasons_df, on='customer_ID', how='left')

scores[['customer_ID', 'display_name', 'gru_top_reasons']].head(3)

,customer_ID,display_name,gru_top_reasons
0,0005ef1480ffe4f0e5908eae07db8e05f2e705bca6a57d...,Asma Marwan Alshamrani,P_2 -> higher risk; D_78_log -> higher risk; S...
1,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,Duha Anas Althunayan,D_41 -> higher risk; P_2 -> higher risk; B_4 -...
2,00130d73f6ebd4c0066dd5ec5c290378ff29d4fc62c177...,Layan Fahad Alhumaidi,P_2 -> lower risk; D_92_log -> lower risk; D_3...


**Insight:** both branches' top-3 reasons are computed once, here, and saved as plain text columns, not recomputed live when the dashboard opens a customer's card. Firasa_04's Step 15 already showed the fusion split between branches can vary a lot per customer (one example was 75% XGBoost-driven, another 59% GRU-driven), so keeping both branches' reasons side by side, rather than picking one, matters for an honest per-customer explanation, exactly the fix that notebook made to its own card design.

## Step 8: Each Customer's Fusion Contribution Split

The exact split between XGBoost and GRU for this specific customer's decision, using the same logit-decomposition Firasa_04's Step 15 used for its 4 example cards, now computed for everyone.

In [11]:
contrib_xgb = meta.coef_[0][0] * logit(p('p_xgb'))
contrib_gru = meta.coef_[0][1] * logit(p('p_branch2'))
total = np.abs(contrib_xgb) + np.abs(contrib_gru)

scores['xgb_contribution_pct'] = np.where(total > 0, np.abs(contrib_xgb) / total, 0.5)
scores['gru_contribution_pct'] = np.where(total > 0, np.abs(contrib_gru) / total, 0.5)

print(scores[['xgb_contribution_pct', 'gru_contribution_pct']].describe().round(3))

       xgb_contribution_pct  gru_contribution_pct
count              8297.000              8297.000
mean                  0.472                 0.528
std                   0.131                 0.131
min                   0.000                 0.001
25%                   0.425                 0.490
50%                   0.459                 0.541
75%                   0.510                 0.575
max                   0.999                 1.000


## Step 9: Assemble and Export the Final Table

One row per test customer, everything the dashboard needs to render a name, a risk badge, and an explanation card without touching a model or recomputing anything live.

In [12]:
export_cols = [
    'customer_ID', 'display_name', 'p_fused', 'risk_level', 'action',
    'xgb_contribution_pct', 'gru_contribution_pct',
    'xgb_top_reasons', 'gru_top_reasons',
]
export_df = scores[export_cols].sort_values('p_fused', ascending=False).reset_index(drop=True)
export_df.to_csv(EXPORT_PATH, index=False)

print(f"Exported {len(export_df)} customers to {EXPORT_PATH}")
export_df.head(10)

Exported 8297 customers to /content/drive/MyDrive/firasa_artifacts/customer_scores_with_shap.csv


,customer_ID,display_name,p_fused,risk_level,action,xgb_contribution_pct,gru_contribution_pct,xgb_top_reasons,gru_top_reasons
0,c22408097c3ac58df97e1cfd6852d765386204b4dfe165...,Noor Ahmed Alkhalidi,0.987690,Critical,Urgent: contact the customer right away and re...,0.599446,0.400554,P_2_last -> higher risk; D_39_last -> higher r...,D_63 -> lower risk; R_6 -> higher risk; R_12_l...
1,4d50f9c53c761bf096d9fd33c7278648b75a0e423614cb...,Kholoud Marwan Almuhaidib,0.985163,Critical,Urgent: contact the customer right away and re...,0.582271,0.417729,P_2_last -> higher risk; D_39_last -> higher r...,D_78 -> lower risk; D_54 -> lower risk; R_8 ->...
2,54f78dfbbd5a019b3e015a61fd74234786a482ec8b668b...,Sultan Munif Alkinani,0.985147,Critical,Urgent: contact the customer right away and re...,0.593891,0.406109,P_2_last -> higher risk; D_39_last -> higher r...,B_41_log -> lower risk; R_2 -> higher risk; P_...
3,d8fd7bda1566238f117265b552a52c9ac31b1fe2562dc2...,Yasser Salem Albalawi,0.985049,Critical,Urgent: contact the customer right away and re...,0.591407,0.408593,P_2_last -> higher risk; D_39_last -> higher r...,D_65 -> higher risk; D_72 -> higher risk; D_54...
4,026c94b49c5ce182f3266812edf12d631f42391946064a...,Aisha Saeed Albalawi,0.984773,Critical,Urgent: contact the customer right away and re...,0.616894,0.383106,P_2_last -> higher risk; D_39_last -> higher r...,S_23_log -> lower risk; R_15_log -> higher ris...
5,f119f9d56b5647968a9ce96ecb80c5b477117fce0da147...,Nayef Majed Altamimi,0.984748,Critical,Urgent: contact the customer right away and re...,0.580222,0.419778,P_2_last -> higher risk; D_39_last -> higher r...,D_84_log -> higher risk; D_65 -> higher risk; ...
6,b9a963ab4418274f82ad4efa1c9b36c2e2e53f3f3710a5...,Dalia Yousef Alkhalidi,0.984711,Critical,Urgent: contact the customer right away and re...,0.587666,0.412334,P_2_last -> higher risk; D_39_last -> higher r...,R_2_log -> higher risk; R_4 -> lower risk; B_2...
7,5f882952bcc7db6c59b02117f3f97ee5428eee7abbfad2...,Safaa Laith Aljuhani,0.984629,Critical,Urgent: contact the customer right away and re...,0.579288,0.420712,P_2_last -> higher risk; D_39_last -> higher r...,R_6_log -> lower risk; D_81 -> higher risk; D_...
8,548d9abb71cb5bdf02bfb555d8261ee00f520fbe9bf3ab...,Safaa Hassan Althunayan,0.984516,Critical,Urgent: contact the customer right away and re...,0.587485,0.412515,P_2_last -> higher risk; D_39_last -> higher r...,R_14 -> lower risk; R_4_log -> higher risk; B_...
9,d31c3fb13b0f3c9bcce7ff76e0a33c4930d636465e1c1d...,Duha Khalid Alasmari,0.984477,Critical,Urgent: contact the customer right away and re...,0.581086,0.418914,P_2_last -> higher risk; D_39_last -> higher r...,D_65 -> higher risk; D_54 -> lower risk; D_54_...


**Insight:** the exported table has one row per test customer (about 8,297 rows), each with a stable display name, a fused probability, a risk level and recommended action, both branches' top reasons, and the exact contribution split for that customer's decision. This is the single file the dashboard's `data/` folder needs; the dashboard never has to load a model, run SHAP, or touch the GRU directly. Re-running this notebook after any change upstream (a retrained branch, a new risk cutoff) regenerates this file the same way, without touching the dashboard code at all.

## Step 10: Monthly Trajectory Export (for the Dashboard's Pulse Chart)

Everything exported so far is one snapshot per customer, the risk at their last statement. The customer profile page needs a pulse: how that risk moved month to month, not just where it ended up.

The GRU is a many-to-many model, it already produces one score per month internally; Firasa_04 only ever read three of those positions (6 months before, 3 months before, last statement) for evaluation. This step reads every real month instead and exports it as its own table, since it is long-format (many rows per customer) rather than the one-row-per-customer shape of the main export.

This is Branch 2's view only, not the fused score. Fusion needs Branch 1's aggregated features recomputed at each point in time, which Firasa_03 only built for 3 checkpoints, not all 13 months, so a fused monthly trajectory isn't available without rebuilding that pipeline. A GRU-only pulse is still the right signal here, since the GRU is the branch that actually reads the sequence month by month; Branch 1 has no month-by-month view to show at all.

In [13]:
pred_full = gru_model.predict(X_test_seq, batch_size=1024, verbose=0)[:, :, 0]   # (n_customers, MAX_LEN)

# Sequences were padded at the front (Step 3), so a customer with L real months
# has their real scores in the LAST L positions; the rest is padding to ignore.
lengths_test = test_df.groupby('customer_ID', sort=True).size().reindex(ids_test).values

trajectory_rows = []
for i, cid in enumerate(ids_test):
    L = lengths_test[i]
    real_scores = pred_full[i, MAX_LEN - L:]
    for month_index, score in enumerate(real_scores, start=1):
        trajectory_rows.append({'customer_ID': cid, 'month_index': month_index, 'risk_score': float(score)})

trajectory_df = pd.DataFrame(trajectory_rows)
trajectory_df = trajectory_df.merge(scores[['customer_ID', 'display_name']], on='customer_ID', how='left')

TRAJECTORY_EXPORT_PATH = f'{ARTIFACT_DIR}/customer_trajectories.csv'
trajectory_df.to_csv(TRAJECTORY_EXPORT_PATH, index=False)

print(f"Exported {trajectory_df['customer_ID'].nunique()} customers, "
      f"{len(trajectory_df)} monthly rows, to {TRAJECTORY_EXPORT_PATH}")
trajectory_df.head(10)

Exported 8297 customers, 99795 monthly rows, to /content/drive/MyDrive/firasa_artifacts/customer_trajectories.csv


,customer_ID,month_index,risk_score,display_name
0,0005ef1480ffe4f0e5908eae07db8e05f2e705bca6a57d...,1,0.937700,Asma Marwan Alshamrani
1,0005ef1480ffe4f0e5908eae07db8e05f2e705bca6a57d...,2,0.967073,Asma Marwan Alshamrani
2,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,1,0.878945,Duha Anas Althunayan
3,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,2,0.925421,Duha Anas Althunayan
4,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,3,0.926578,Duha Anas Althunayan
5,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,4,0.926621,Duha Anas Althunayan
6,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,5,0.917281,Duha Anas Althunayan
7,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,6,0.964912,Duha Anas Althunayan
8,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,7,0.966767,Duha Anas Althunayan
9,000de143ca96be8fa6362e55dffe94a0bdaf347d993160...,8,0.971411,Duha Anas Althunayan


**Insight:** each customer contributes as many rows as they have real statements (1 to 13), so the total row count is close to `n_statements` summed across all test customers, not a fixed 8,297 x 13. `month_index` starts at 1 for a customer's earliest statement and counts up to their last one, so the dashboard can plot it directly as a line chart without any further reshaping. This file goes in the dashboard's `data/` folder alongside `customer_scores_with_shap.csv`; the profile page reads both, one for the snapshot, one for the pulse.